# Phase 5 — Model Training

Train three classifiers using the cleaned dataset and a reusable preprocessing pipeline.

Models:
- Logistic Regression
- Decision Tree
- Random Forest

In [1]:
from pathlib import Path
import json
import joblib
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

In [2]:
ROOT = Path.cwd().parent if Path.cwd().name=="notebooks" else Path.cwd()

DATA = ROOT/"data/processed/clean_employee_attrition.csv"
MODELS = ROOT/"models"
REPORTS = ROOT/"reports"

MODELS.mkdir(exist_ok=True)
REPORTS.mkdir(exist_ok=True)

df = pd.read_csv(DATA)

TARGET="Attrition"
X=df.drop(columns=[TARGET])
y=df[TARGET]

In [3]:
num_cols=X.select_dtypes(include=np.number).columns.tolist()
cat_cols=X.select_dtypes(exclude=np.number).columns.tolist()

numeric_pipe=Pipeline([
    ("imputer",SimpleImputer(strategy="median")),
    ("scaler",StandardScaler())
])

categorical_pipe=Pipeline([
    ("imputer",SimpleImputer(strategy="most_frequent")),
    ("encoder",OneHotEncoder(handle_unknown="ignore"))
])

preprocessor=ColumnTransformer([
    ("num",numeric_pipe,num_cols),
    ("cat",categorical_pipe,cat_cols)
])

X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,
    random_state=42,
    stratify=y
)

## Logistic Regression

In [4]:
logistic_pipeline=Pipeline([
    ("preprocessor",preprocessor),
    ("model",LogisticRegression(max_iter=1000,random_state=42))
])

logistic_pipeline.fit(X_train,y_train)

print("Logistic Regression trained.")

Logistic Regression trained.


## Decision Tree

In [5]:
tree_pipeline=Pipeline([
    ("preprocessor",preprocessor),
    ("model",DecisionTreeClassifier(random_state=42))
])

tree_pipeline.fit(X_train,y_train)

print("Decision Tree trained.")

Decision Tree trained.


## Random Forest

In [6]:
forest_pipeline=Pipeline([
    ("preprocessor",preprocessor),
    ("model",RandomForestClassifier(
        n_estimators=200,
        random_state=42
    ))
])

forest_pipeline.fit(X_train,y_train)

print("Random Forest trained.")

Random Forest trained.


## Save Models

In [7]:
joblib.dump(logistic_pipeline,MODELS/"logistic_regression.joblib")
joblib.dump(tree_pipeline,MODELS/"decision_tree.joblib")
joblib.dump(forest_pipeline,MODELS/"random_forest.joblib")

metadata={
    "models":[
        "logistic_regression.joblib",
        "decision_tree.joblib",
        "random_forest.joblib"
    ],
    "train_rows":len(X_train),
    "test_rows":len(X_test),
    "random_state":42,
    "test_size":0.2
}

with open(REPORTS/"training_metadata.json","w") as f:
    json.dump(metadata,f,indent=4)

print("Models saved successfully.")

Models saved successfully.


# Outputs

```
models/
├── logistic_regression.joblib
├── decision_tree.joblib
└── random_forest.joblib

reports/
└── training_metadata.json
```

